# ABSA Training Pipeline v3 — RoBERTa-base | 8 Nhãn
**Aspect-Based Sentiment Analysis** | Khóa luận tốt nghiệp

## Thay đổi so với v2 (fix over-correction)

| Ô | Thay đổi | Lý do |
|---|----------|-------|
| 2 | `pos_weight = sqrt(N_neg/N_pos)` thay vì hard ratio | Hard ratio ~51x gây gradient explosion |
| 4 | **Bỏ WeightedRandomSampler** — dùng shuffle=True | Triple stacking (sampler+pos_weight+focal) = over-correction |
| 6 | `alpha` clip mở rộng lên [0.25, 0.75] | pos_weight đã soft → alpha cần phạt đúng hơn |
| 8 | `EPOCHS=15`, `PATIENCE=5`, monitor=`0.5*macro+0.5*micro` | Model chưa converge ở E10 |

**Root cause:** `pos_weight(hard≈50x) × alpha_focal × (1-p)^gamma` = triple penalty  
→ Common classes bị học kém → Micro-F1 giảm → Macro-F1 giảm theo


In [14]:
# Ô 1 — Cài đặt & Import
!pip install transformers scikit-learn torch emoji --quiet

import gc, os, json, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaModel,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, classification_report,
    precision_score, recall_score, fbeta_score,
)

# ── Global random seed ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print(f'Seed set: {SEED}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.6 MB/s eta 0:00:00
Seed set: 42
Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [2]:
# Ô 2 — Cấu hình
from google.colab import drive
drive.mount('/content/drive')

import gdown, os

def gdown_id(url, out):
    file_id = url.split('/d/')[1].split('/')[0]
    gdown.download(f'https://drive.google.com/uc?id={file_id}', out, quiet=False)

gdown_id('https://drive.google.com/file/d/1yQ1HDrXFX8tEqaE5rpogcQleoeyZ-YCF/view?usp=drive_link', '/content/train.csv')
gdown_id('https://drive.google.com/file/d/1Klhao1HSfoBpeWAl6ILMfYdWYgeeywhI/view?usp=drive_link', '/content/val.csv')
gdown_id('https://drive.google.com/file/d/1hOLCW1nmapR3pQA5iAzszbDHVhF94U1R/view?usp=drive_link', '/content/test.csv')
gdown_id('https://drive.google.com/file/d/1Z2TnHk6DDvcPxqTOttsnM2FwZQ9QfFA0/view?usp=drive_link', '/content/pos_weight.json')

TRAIN_PATH  = '/content/train.csv'
VAL_PATH    = '/content/val.csv'
TEST_PATH   = '/content/test.csv'
POS_W_PATH  = '/content/pos_weight.json'
SAVE_PATH   = '/content/best_absa_v3.pt'
THRESH_PATH = '/content/best_thresholds_v3.npy'
DRIVE_DIR   = '/content/drive/MyDrive/KLTN/ketquamohinh/'

os.makedirs(DRIVE_DIR, exist_ok=True)

LABEL_COLS = [
    'scent_positive',    'scent_negative',
    'longevity_positive','longevity_negative',
    'packaging_positive','packaging_negative',
    'service_positive',  'service_negative',
]
NUM_LABELS  = len(LABEL_COLS)
ASPECTS     = ['scent', 'longevity', 'packaging', 'service']

# ── Hyperparams ──────────────────────────────────────────────────────────────
EPOCHS      = 15      # Tăng từ 10 → 15: model chưa converge ở E10
BATCH_SIZE  = 8
ACCUM_STEPS = 8
MAX_LEN     = 128     # Sẽ tự adjust ở Ô 3
LR_HEAD     = 1e-4
LR_TOP      = 3e-5
LR_DECAY    = 0.85
PATIENCE    = 5       # Tăng từ 3 → 5: ổn định hơn với noisy Macro-F1

# ── Load pos_weight — SOFT version: sqrt(N_neg/N_pos) ────────────────────────
# Lý do: hard ratio (N_neg/N_pos) lên tới ~51x gây triple-penalty
# khi kết hợp với focal-alpha và focal-gamma → common classes bị nhiễu
# sqrt() giữ chiều tác động đúng nhưng giảm magnitude (51x → 7x)
with open(POS_W_PATH, 'r') as _f:
    _pw_dict = json.load(_f)

POS_WEIGHT = torch.tensor(
    [max(1.0, _pw_dict[c] ** 0.5) for c in LABEL_COLS],   # SQRT soft weighting
    dtype=torch.float
).to(device)

print(f'NUM_LABELS : {NUM_LABELS}')
print(f'Labels     : {LABEL_COLS}')
print(f'Output dir : {DRIVE_DIR}')
print('\npos_weight per label (sqrt-soft, clipped >= 1):')
print(f'  {"Label":<24} {"pw_raw":>8} {"pw_sqrt":>9}')
print('  ' + '-' * 45)
for col in LABEL_COLS:
    raw = _pw_dict[col]
    soft = max(1.0, raw ** 0.5)
    print(f'  {col:<24} {raw:>8.3f} {soft:>9.3f}')

Mounted at /content/drive


Downloading...
From: https://drive.google.com/uc?id=1yQ1HDrXFX8tEqaE5rpogcQleoeyZ-YCF
To: /content/train.csv
100%|██████████| 8.32M/8.32M [00:00<00:00, 249MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Klhao1HSfoBpeWAl6ILMfYdWYgeeywhI
To: /content/val.csv
100%|██████████| 1.77M/1.77M [00:00<00:00, 150MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hOLCW1nmapR3pQA5iAzszbDHVhF94U1R
To: /content/test.csv
100%|██████████| 1.77M/1.77M [00:00<00:00, 181MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Z2TnHk6DDvcPxqTOttsnM2FwZQ9QfFA0
To: /content/pos_weight.json
100%|██████████| 246/246 [00:00<00:00, 296kB/s]


NUM_LABELS : 8
Labels     : ['scent_positive', 'scent_negative', 'longevity_positive', 'longevity_negative', 'packaging_positive', 'packaging_negative', 'service_positive', 'service_negative']
Output dir : /content/drive/MyDrive/KLTN/ketquamohinh/

pos_weight per label (sqrt-soft, clipped >= 1):
  Label                      pw_raw   pw_sqrt
  ---------------------------------------------
  scent_positive              1.000     1.000
  scent_negative             19.099     4.370
  longevity_positive          1.639     1.280
  longevity_negative         21.410     4.627
  packaging_positive          3.070     1.752
  packaging_negative         50.231     7.087
  service_positive           27.003     5.196
  service_negative           48.860     6.990


In [3]:
# Ô 3 — Load Data + Tự chọn MAX_LEN
df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {len(df_train):,}")
print(f"Val   : {len(df_val):,}")
print(f"Test  : {len(df_test):,}")

X_train = df_train["full_text"].tolist()
y_train = df_train[LABEL_COLS].values.astype(float).tolist()
X_val   = df_val["full_text"].tolist()
y_val   = df_val[LABEL_COLS].values.astype(float).tolist()
X_test  = df_test["full_text"].tolist()
y_test  = df_test[LABEL_COLS].values.astype(float).tolist()

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
_lengths  = [len(tokenizer.encode(t, truncation=False)) for t in X_train[:3000]]
_pct      = np.mean(np.array(_lengths) > 128) * 100
MAX_LEN     = 256 if _pct > 15 else 128
BATCH_SIZE  = 8   if MAX_LEN == 256 else 16
ACCUM_STEPS = 8   if MAX_LEN == 256 else 4

print(f"\nToken mean={np.mean(_lengths):.0f}  P90={np.percentile(_lengths, 90):.0f}")
print(f"Bị cắt ở 128: {_pct:.1f}%")
print(f"→ MAX_LEN={MAX_LEN} | batch={BATCH_SIZE} | accum={ACCUM_STEPS} | eff_batch={BATCH_SIZE*ACCUM_STEPS}")

# Tần suất nhãn
label_array = np.array(y_train)
label_freq  = label_array.sum(axis=0)
label_freq  = np.where(label_freq == 0, 1, label_freq)

print("\nPhân bố nhãn (train):")
for i, col in enumerate(LABEL_COLS):
    print(f"  {col:<24} {int(label_freq[i]):>6}  ({label_freq[i]/len(y_train)*100:.2f}%)")

Train : 32,592
Val   : 6,982
Test  : 6,995


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors



Token mean=168  P90=318
Bị cắt ở 128: 56.0%
→ MAX_LEN=256 | batch=8 | accum=8 | eff_batch=64

Phân bố nhãn (train):
  scent_positive            29741  (91.25%)
  scent_negative             1621  (4.97%)
  longevity_positive        12351  (37.90%)
  longevity_negative         1454  (4.46%)
  packaging_positive         8009  (24.57%)
  packaging_negative          637  (1.95%)
  service_positive           1164  (3.57%)
  service_negative            654  (2.01%)


In [4]:
# Ô 4 — Dataset & DataLoader (KHÔNG dùng WeightedRandomSampler)
#
# Lý do bỏ Sampler:
#   Bản v2 dùng đồng thời 3 cơ chế imbalance correction:
#     1) WeightedRandomSampler (oversampling)
#     2) pos_weight trong BCE
#     3) alpha/gamma trong Focal Loss
#   → Triple-stacking gây over-correction:
#        common labels (scent_positive ~91%) bị học kém
#        → Micro-F1 giảm từ 0.9027 → 0.8248
#   DOMAIN-REVIEW M2 gợi ý: "focal loss alone often suffices"
#   → Giữ pos_weight(soft) + focal là đủ

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float),
        }


train_ds = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = ReviewDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# shuffle=True thay cho sampler: natural distribution + pos_weight xử lý imbalance
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          generator=g, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print('\nSampler: shuffle=True (natural dist) | Imbalance → pos_weight + focal')

Train batches : 4074
Val   batches : 437

Sampler: shuffle=True (natural dist) | Imbalance → pos_weight + focal


In [5]:
# Ô 5 — Model (CLS + Attention Pooling) — giữ nguyên
class ABSAModel(nn.Module):
    """
    RoBERTa-base + CLS & Attention Pooling → concat(1536) → classifier.
    """
    def __init__(self, num_labels: int = 8, dropout: float = 0.2):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained("roberta-base")
        hidden          = self.roberta.config.hidden_size  # 768
        self.attn_pool  = nn.Linear(hidden, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_labels),
        )

    def _attn_pool(self, hidden, mask):
        w = self.attn_pool(hidden).squeeze(-1)
        w = w.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(w, dim=-1)
        return (hidden * w.unsqueeze(-1)).sum(1)

    def forward(self, input_ids, attention_mask):
        out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls    = out.last_hidden_state[:, 0, :]
        attn   = self._attn_pool(out.last_hidden_state, attention_mask)
        pooled = torch.cat([cls, attn], dim=-1)
        return self.classifier(pooled)


model = ABSAModel(num_labels=NUM_LABELS).to(device)
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Params: 125.4M


In [6]:
# Ô 6 — Focal Loss Adaptive v3
#
# Thay đổi so với v2:
#   - alpha clip mở rộng: [0.25, 0.75] thay vì [0.20, 0.45]
#     Lý do: pos_weight đã soft (sqrt) → alpha cần phạt thêm cho rare classes
#     alpha=0.75 cho rare class: positive mới thực sự nặng hơn negative
#   - pos_weight đưa vào như v2 (đã đọc từ file)
#   - label smoothing nhất quán (smooth_t cho cả p_t và alpha_t)

class FocalLossAdaptive(nn.Module):
    """
    Gamma & Alpha khác nhau per-label theo tần suất nhãn.
    - Nhãn hiếm → gamma cao hơn + alpha cao hơn.
    - pos_weight SOFT (sqrt) tích hợp vào BCE.
    - Label smoothing (eps=0.05) áp dụng nhất quán.
    - alpha clip [0.25, 0.75]: rare class nhận alpha > 0.5 (positive majority).
    """
    def __init__(self, label_freq, pos_weight,
                 base_gamma=2.0, max_gamma=5.0,
                 base_alpha=0.25, label_smoothing=0.05):
        super().__init__()
        self.ls       = label_smoothing
        freq_norm     = label_freq / label_freq.max()
        gammas        = base_gamma + (max_gamma - base_gamma) * (1 - freq_norm)

        # alpha [0.25, 0.75]: rare class có alpha > 0.5
        # → positive examples của rare class được up-weight so với negative
        alphas        = np.clip(base_alpha + 0.50 * (1 - freq_norm), 0.25, 0.75)

        self.register_buffer('gammas',     torch.tensor(gammas,     dtype=torch.float))
        self.register_buffer('alphas',     torch.tensor(alphas,     dtype=torch.float))
        self.register_buffer('pos_weight', pos_weight.float())

    def forward(self, logits, targets):
        # Label smoothing nhất quán cho cả BCE và focal modulation
        smooth_t = targets * (1 - self.ls) + 0.5 * self.ls
        probs    = torch.sigmoid(logits)

        # BCE với pos_weight soft
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, smooth_t,
            pos_weight=self.pos_weight,
            reduction='none',
        )

        # p_t & alpha_t dùng smooth_t (nhất quán)
        p_t     = probs * smooth_t + (1 - probs) * (1 - smooth_t)
        alpha_t = self.alphas * smooth_t + (1 - self.alphas) * (1 - smooth_t)
        return (alpha_t * (1 - p_t) ** self.gammas * bce).mean()


criterion = FocalLossAdaptive(label_freq, POS_WEIGHT).to(device)

print('Gamma, Alpha, pos_weight (sqrt-soft) per label:')
print(f'  {"Label":<24} {"gamma":>6} {"alpha":>7} {"pos_w":>8}')
print('  ' + '-' * 52)
for i, col in enumerate(LABEL_COLS):
    print(f'  {col:<24} {criterion.gammas[i]:>6.2f} '
          f'{criterion.alphas[i]:>7.3f} '
          f'{criterion.pos_weight[i]:>8.3f}')

Gamma, Alpha, pos_weight (sqrt-soft) per label:
  Label                     gamma   alpha    pos_w
  ----------------------------------------------------
  scent_positive             2.00   0.250    1.000
  scent_negative             4.84   0.723    4.370
  longevity_positive         3.75   0.542    1.280
  longevity_negative         4.85   0.726    4.627
  packaging_positive         4.19   0.615    1.752
  packaging_negative         4.94   0.739    7.087
  service_positive           4.88   0.730    5.196
  service_negative           4.93   0.739    6.990


In [7]:
# Ô 7 — Optimizer (LLRD) + Scheduler — giữ nguyên
def get_llrd_params(model, base_lr=LR_TOP, decay=LR_DECAY, head_lr=LR_HEAD):
    """Layer-wise LR Decay: layer trên → LR cao hơn."""
    groups = []
    groups.append({
        "params": list(model.classifier.parameters()) + list(model.attn_pool.parameters()),
        "lr": head_lr, "weight_decay": 0.01, "name": "head",
    })
    layers = model.roberta.encoder.layer
    for i in range(len(layers) - 1, -1, -1):
        groups.append({
            "params": layers[i].parameters(),
            "lr": base_lr * (decay ** (len(layers) - 1 - i)),
            "weight_decay": 0.01, "name": f"layer_{i}",
        })
    groups.append({
        "params": model.roberta.embeddings.parameters(),
        "lr": base_lr * (decay ** len(layers)),
        "weight_decay": 0.01, "name": "embeddings",
    })
    return groups


optimizer    = torch.optim.AdamW(get_llrd_params(model), eps=1e-8)
total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * 0.10)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler("cuda")

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")
print("\nLR per group (top 3):")
for g in get_llrd_params(model)[:3]:
    print(f"  {g['name']:<20} lr={g['lr']:.2e}")
print(f"  ...")

Total steps  : 7635
Warmup steps : 763

LR per group (top 3):
  head                 lr=1.00e-04
  layer_11             lr=3.00e-05
  layer_10             lr=2.55e-05
  ...


In [8]:
# Ô 8 — Training Loop v3
#
# Thay đổi so với v2:
#   - monitor_score = 0.5*macro + 0.5*micro
#     Lý do: Macro-F1 trên val rất noisy với imbalanced data
#     Combined score ổn định hơn → early stopping không bị trigger sớm
#     Nhưng vẫn ưu tiên macro (50%) đúng như yêu cầu review
#   - PATIENCE=5 (từ 3): cho model thêm thời gian với noisy metric
#   - EPOCHS=15: model v2 chưa converge ở E10

best_score       = 0.0
patience_counter = 0
history          = []

for epoch in range(EPOCHS):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)

        with torch.amp.autocast('cuda'):
            loss = criterion(model(ids, mask), lbls) / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % 300 == 0:
            print(f'  E{epoch+1} step {step+1}/{len(train_loader)} '
                  f'loss={total_loss/(step+1):.4f} '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e}')

    # Flush gradient tích lũy cuối epoch
    optimizer.zero_grad()

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    vp_all, vl_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask)
            vp_all.append((torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int))
            vl_all.append(batch['labels'].numpy())

    vp = np.vstack(vp_all);  vl = np.vstack(vl_all)
    micro    = f1_score(vl, vp, average='micro', zero_division=0)
    macro    = f1_score(vl, vp, average='macro', zero_division=0)
    avg_loss = total_loss / len(train_loader)

    # Early stopping: combined 0.5*macro + 0.5*micro
    # Macro-F1 là metric chính (thesis), nhưng combined ổn định hơn
    # trên val imbalanced (tránh early stop do noise)
    monitor_score = 0.5 * macro + 0.5 * micro

    print(f'\n{"="*65}')
    print(f'Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | '
          f'Micro={micro:.4f} | Macro={macro:.4f} | Monitor={monitor_score:.4f}')
    print(f'{"="*65}')

    history.append({'epoch': epoch+1, 'loss': avg_loss,
                    'micro': micro, 'macro': macro, 'monitor': monitor_score})

    if monitor_score > best_score:
        best_score = monitor_score
        patience_counter = 0
        torch.save({
            'epoch':        epoch + 1,
            'model_state':  model.state_dict(),
            'val_micro':    micro,
            'val_macro':    macro,
            'val_monitor':  monitor_score,
            'max_len':      MAX_LEN,
            'label_cols':   LABEL_COLS,
        }, SAVE_PATH)
        print(f'✅ Saved | Macro={macro:.4f} | Micro={micro:.4f} | Monitor={monitor_score:.4f}\n')
    else:
        patience_counter += 1
        print(f'⚠️  No improvement {patience_counter}/{PATIENCE} '
              f'(best Monitor={best_score:.4f})\n')
        if patience_counter >= PATIENCE:
            print('🛑 Early stopping!')
            break

    gc.collect(); torch.cuda.empty_cache()

# Report
best_epoch = max(history, key=lambda h: h['monitor'])
print(f'\n🏆 Best checkpoint: Epoch {best_epoch["epoch"]} | '
      f'Macro={best_epoch["macro"]:.4f} | Micro={best_epoch["micro"]:.4f}')
print('\nHistory:')
print(f'  {"E":>3} {"loss":>8} {"micro":>8} {"macro":>8} {"monitor":>9}')
for h in history:
    marker = " ← best" if h['epoch'] == best_epoch['epoch'] else ""
    print(f'  E{h["epoch"]:02d} {h["loss"]:>8.4f} {h["micro"]:>8.4f} '
          f'{h["macro"]:>8.4f} {h["monitor"]:>9.4f}{marker}')

  E1 step 300/4074 loss=0.0339 lr=4.85e-06
  E1 step 600/4074 loss=0.0294 lr=9.83e-06
  E1 step 900/4074 loss=0.0266 lr=1.47e-05
  E1 step 1200/4074 loss=0.0247 lr=1.97e-05
  E1 step 1500/4074 loss=0.0235 lr=2.45e-05
  E1 step 1800/4074 loss=0.0227 lr=2.95e-05
  E1 step 2100/4074 loss=0.0221 lr=3.43e-05
  E1 step 2400/4074 loss=0.0215 lr=3.93e-05
  E1 step 2700/4074 loss=0.0210 lr=4.42e-05
  E1 step 3000/4074 loss=0.0207 lr=4.91e-05
  E1 step 3300/4074 loss=0.0202 lr=5.40e-05
  E1 step 3600/4074 loss=0.0198 lr=5.90e-05
  E1 step 3900/4074 loss=0.0194 lr=6.38e-05

Epoch 1/15 | loss=0.0192 | Micro=0.7835 | Macro=0.4741 | Monitor=0.6288
✅ Saved | Macro=0.4741 | Micro=0.7835 | Monitor=0.6288

  E2 step 300/4074 loss=0.0149 lr=7.16e-05
  E2 step 600/4074 loss=0.0143 lr=7.65e-05
  E2 step 900/4074 loss=0.0139 lr=8.14e-05
  E2 step 1200/4074 loss=0.0137 lr=8.64e-05
  E2 step 1500/4074 loss=0.0136 lr=9.12e-05
  E2 step 1800/4074 loss=0.0133 lr=9.62e-05
  E2 step 2100/4074 loss=0.0131 lr=9.99e-

In [9]:
# Ô 9 — Threshold Tuning (Val Set) — giữ nguyên logic
ckpt = torch.load(SAVE_PATH)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded | Epoch {ckpt['epoch']} | "
      f"Micro={ckpt['val_micro']:.4f} | Macro={ckpt['val_macro']:.4f}\n")

probs_val, labels_val = [], []
with torch.no_grad():
    for batch in val_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.amp.autocast("cuda"):
            logits = model(ids, mask)
        probs_val.append(torch.sigmoid(logits).cpu().numpy())
        labels_val.append(batch["labels"].numpy())

probs_val  = np.vstack(probs_val)
labels_val = np.vstack(labels_val)

RARE_SUPPORT    = 200
best_thresholds = np.zeros(NUM_LABELS)

print(f"{'Label':<25} {'T':>6} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Sup':>6}")
print("-" * 58)

for i, col in enumerate(LABEL_COLS):
    support = int(labels_val[:, i].sum())
    is_rare = support < RARE_SUPPORT
    t_range = np.arange(0.05, 0.61, 0.005) if is_rare else np.arange(0.10, 0.91, 0.005)
    beta    = 1.5 if is_rare else 1.0
    min_rec = 0.20 if is_rare else 0.10

    best_t, best_fb = 0.5, 0.0
    for t in t_range:
        preds = (probs_val[:, i] > t).astype(int)
        rec   = recall_score(labels_val[:, i], preds, zero_division=0)
        if support > 0 and rec < min_rec:
            continue
        fb = fbeta_score(labels_val[:, i], preds, beta=beta, zero_division=0)
        if fb > best_fb:
            best_fb, best_t = fb, t

    best_thresholds[i] = best_t
    preds = (probs_val[:, i] > best_t).astype(int)
    p = precision_score(labels_val[:, i], preds, zero_division=0)
    r = recall_score(labels_val[:, i],    preds, zero_division=0)
    f = f1_score(labels_val[:, i],        preds, zero_division=0)
    print(f"{col:<25} {best_t:>6.3f} {p:>7.3f} {r:>7.3f} {f:>7.4f} {support:>6}"
          + (" ← rare" if is_rare else ""))

np.save(THRESH_PATH, best_thresholds)
print(f"\n✅ Saved thresholds → {THRESH_PATH}")

pf = (probs_val > 0.5).astype(int)
pt = (probs_val > best_thresholds).astype(int)
print(f"\n{'Metric':<20} {'Fixed(0.5)':>12} {'Tuned':>12}")
print("-" * 46)
print(f"{'Micro-F1':<20} "
      f"{f1_score(labels_val, pf, average='micro', zero_division=0):>12.4f} "
      f"{f1_score(labels_val, pt, average='micro', zero_division=0):>12.4f}")
print(f"{'Macro-F1':<20} "
      f"{f1_score(labels_val, pf, average='macro', zero_division=0):>12.4f} "
      f"{f1_score(labels_val, pt, average='macro', zero_division=0):>12.4f}")

Loaded | Epoch 15 | Micro=0.8727 | Macro=0.6679

Label                          T    Prec     Rec      F1    Sup
----------------------------------------------------------
scent_positive             0.225   0.964   0.992  0.9778   6375
scent_negative             0.600   0.724   0.724  0.7241    348
longevity_positive         0.540   0.920   0.896  0.9076   2647
longevity_negative         0.650   0.836   0.670  0.7438    312
packaging_positive         0.585   0.899   0.847  0.8726   1717
packaging_negative         0.600   0.472   0.551  0.5085    136 ← rare
service_positive           0.635   0.603   0.456  0.5194    250
service_negative           0.595   0.675   0.743  0.7075    140 ← rare

✅ Saved thresholds → /content/best_thresholds_v3.npy

Metric                 Fixed(0.5)        Tuned
----------------------------------------------
Micro-F1                   0.8727       0.9170
Macro-F1                   0.6679       0.7451


In [ ]:
# Ô 10 — Đánh giá Test Set + Lưu Drive
from google.colab import drive
from datetime import datetime
import shutil

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

model.eval()
probs_test, labels_test = [], []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.amp.autocast("cuda"):
            logits = model(ids, mask)
        probs_test.append(torch.sigmoid(logits).cpu().numpy())
        labels_test.append(batch["labels"].numpy())

probs_test  = np.vstack(probs_test)
labels_test = np.vstack(labels_test)

preds_fixed = (probs_test > 0.5).astype(int)
preds_tuned = (probs_test > best_thresholds).astype(int)

micro_f = f1_score(labels_test, preds_fixed, average="micro", zero_division=0)
macro_f = f1_score(labels_test, preds_fixed, average="macro", zero_division=0)
micro_t = f1_score(labels_test, preds_tuned, average="micro", zero_division=0)
macro_t = f1_score(labels_test, preds_tuned, average="macro", zero_division=0)

print(f"\n{'='*60}")
print(f"KẾT QUẢ CUỐI CÙNG — TEST SET")
print(f"{'='*60}")
print(f"{'Config':<35} {'Micro-F1':>10} {'Macro-F1':>10}")
print(f"{'-'*60}")
print(f"{'Threshold = 0.5':<35} {micro_f:>10.4f} {macro_f:>10.4f}")
print(f"{'Per-label tuned':<35} {micro_t:>10.4f} {macro_t:>10.4f}")
print(f"{'='*60}")
print(f"\nSo sánh với baseline:")
print(f"  Bản cũ (có bug):    Micro=0.9027 | Macro=0.7116")
print(f"  Bản v2 (over-fix):  Micro=0.8248 | Macro=0.6173")
print(f"  Bản v3 (này):       Micro={micro_t:.4f} | Macro={macro_t:.4f}")

print("\n=== Classification Report (Tuned) ===\n")
print(classification_report(labels_test, preds_tuned,
                             target_names=LABEL_COLS, zero_division=0))

ts = datetime.now().strftime("%Y%m%d_%H%M")
pd.DataFrame(
    classification_report(labels_test, preds_tuned, target_names=LABEL_COLS,
                          zero_division=0, output_dict=True)
).T.round(4).to_csv(f"{DRIVE_DIR}/test_report_v3_{ts}.csv")

with open(f"{DRIVE_DIR}/summary_v3_{ts}.txt", "w") as f:
    f.write(f"ABSA RoBERTa v3 | 8 nhãn | {datetime.now()}\n")
    f.write(f"Changes: no sampler, pos_weight=sqrt, alpha=[0.25,0.75], epochs=15\n\n")
    f.write(f"Epoch: {ckpt['epoch']} | Val Micro={ckpt['val_micro']:.4f} | Macro={ckpt['val_macro']:.4f}\n\n")
    f.write(f"Test (fixed 0.5): Micro={micro_f:.4f} | Macro={macro_f:.4f}\n")
    f.write(f"Test (tuned)    : Micro={micro_t:.4f} | Macro={macro_t:.4f}\n\n")
    f.write(classification_report(labels_test, preds_tuned,
                                   target_names=LABEL_COLS, zero_division=0))

shutil.copy(SAVE_PATH,   f"{DRIVE_DIR}/best_absa_v3_{ts}.pt")
shutil.copy(THRESH_PATH, f"{DRIVE_DIR}/best_thresholds_v3_{ts}.npy")
print(f"\n✅ Tất cả đã lưu vào {DRIVE_DIR}")

In [15]:
# Ô 11 — Inference (nhất quán với preprocessing)
import re
import emoji as _emoji_lib


def clean_text(text: str) -> str:
    """Chuẩn hoá text -- nhất quán với preprocessing.ipynb.
    Giữ digits (0-9) để bảo toàn signal longevity (8 hours, 30 days...).
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = _emoji_lib.replace_emoji(text, replace='')
    text = re.sub(r"[^a-z0-9\s.,!?;:\-\'\"]", ' ', text)   # giữ digits
    text = re.sub(r'\s+([.,!?;])', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


SENTIMENTS = ['positive', 'negative']
LABEL2ID   = {l: i for i, l in enumerate(LABEL_COLS)}


def predict(text: str) -> dict:
    model.eval()
    cleaned = clean_text(text)
    enc = tokenizer(cleaned, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad(), torch.amp.autocast('cuda'):
        probs = torch.sigmoid(
            model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        ).cpu().numpy()[0]

    result = {}
    for aspect in ASPECTS:
        best_sent, best_conf = 'none', 0.0
        for sent in SENTIMENTS:
            idx  = LABEL2ID[f'{aspect}_{sent}']
            conf = float(probs[idx])
            if conf > best_thresholds[idx] and conf > best_conf:
                best_sent, best_conf = sent, conf
        result[aspect] = {'sentiment': best_sent, 'confidence': round(best_conf, 4)}
    return result


demo = [
    'The scent is absolutely divine and lasts all day long. Packaging was damaged on arrival.',
    'Fades within 2 hours, very disappointing for the price. Only lasted 30 minutes on me.',
    'Great customer service, they replaced my broken bottle immediately!',
]

print('=== INFERENCE DEMO ===')
for review in demo:
    result = predict(review)
    active = {a: v for a, v in result.items() if v['sentiment'] != 'none'}
    print(f'\nReview (raw)   : {review[:80]}')
    print(f'Review (clean) : {clean_text(review)[:80]}')
    if active:
        for aspect, info in active.items():
            bar = chr(9608) * int(info['confidence'] * 20)
            print(f'  {aspect:<12} | {info["sentiment"]:<10} | {info["confidence"]:.3f} {bar}')
    else:
        print('  -> Khong phat hien aspect/sentiment ro rang')

=== INFERENCE DEMO ===

Review (raw)   : The scent is absolutely divine and lasts all day long. Packaging was damaged on 
Review (clean) : the scent is absolutely divine and lasts all day long. packaging was damaged on 
  scent        | positive   | 0.975 ███████████████████
  longevity    | positive   | 0.942 ██████████████████
  packaging    | positive   | 0.806 ████████████████

Review (raw)   : Fades within 2 hours, very disappointing for the price. Only lasted 30 minutes o
Review (clean) : fades within 2 hours, very disappointing for the price. only lasted 30 minutes o
  scent        | positive   | 0.294 █████
  longevity    | negative   | 0.909 ██████████████████

Review (raw)   : Great customer service, they replaced my broken bottle immediately!
Review (clean) : great customer service, they replaced my broken bottle immediately!
  service      | negative   | 0.798 ███████████████
